In [ ]:
print('===== NATIVE MERGEHBHEAD BITEXACT TEST =====')
# 1/4 - clone main + prototype (BiasAddRows .tkv)
import subprocess, os
print('===== NATIVE BIASADD BITEXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (has BiasAddRows) + .tkv-generated C .so
import subprocess, os
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/merge_hb_head_cpu.cpu.c'; so=ex+'/merge_hb_head_cpu.so'
p2=subprocess.run('gcc -O2 -mavx2 -mfma -ffp-contract=off -fno-fast-math -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare MergeHbHead (pure copy)
import ctypes, numpy as np
args=[ctypes.POINTER(ctypes.c_float)]*4+[ctypes.c_int]*5
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').MergeHbHead
native.argtypes=args; native.restype=ctypes.c_int
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/merge_hb_head_cpu.so').MergeHbHead
tkv.argtypes=args; tkv.restype=ctypes.c_int
rng=np.random.default_rng(7)
for n,h,hd,hlo,hhi in [(4,768,64,0,3),(8,768,64,2,6),(2,256,64,0,4)]:
    co=rng.standard_normal(n*hd).astype(np.float32); dq=rng.standard_normal(n*3*hd).astype(np.float32)
    a_ctx=np.zeros(n*h,np.float32); a_qkv=np.zeros(n*3*h,np.float32)
    c_ctx=np.zeros(n*h,np.float32); c_qkv=np.zeros(n*3*h,np.float32)
    pa=[x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)) for x in [co,dq,a_ctx,a_qkv]]
    pc=[x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)) for x in [co,dq,c_ctx,c_qkv]]
    native(*(pa+[n,h,hd,hlo,hhi])); tkv(*(pc+[n,h,hd,hlo,hhi]))
    d1=np.count_nonzero(a_ctx.view(np.uint32)!=c_ctx.view(np.uint32))
    d2=np.count_nonzero(a_qkv.view(np.uint32)!=c_qkv.view(np.uint32))
    print('n=%d h=%d hlo=%d hhi=%d bitwise=%s diff=%d/%d'%(n,h,hlo,hhi,d1+d2==0,d1+d2,a_ctx.size+a_qkv.size))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> BiasAddRows can be replaced by .tkv-generated C.')